In [20]:
from __future__ import annotations

from abc import ABC, abstractmethod
from pathlib import Path
from typing import Callable, Any
from datetime import datetime

import numpy as np
import pandas as pd
import mlflow
import matplotlib.pyplot as plt
import sklearn.metrics as sk_metrics
import shap
import optuna
import yaml
import json

from pftnc.config import AppConfig, DatasetArtifact, DatasetMetadata
from pftnc.utils.utils import get_registry_key_and_paths

In [2]:
mlflow.set_tracking_uri("http://127.0.0.1:8081")

In [3]:
METRIC_REGISTRY: dict[str, Callable[[np.ndarray, np.ndarray], float]] = {
    "mse": sk_metrics.mean_squared_error,
    "mae": sk_metrics.mean_absolute_error,
    "r2": sk_metrics.r2_score,
    "rmse": sk_metrics.root_mean_squared_error
}

In [4]:
class ModelAdapter(ABC):
    """
    Abstract interface for pluggable model backends.
 
    Every concrete adapter must implement all abstract methods so the
    `Trainer` can call them without knowing the underlying library.
    """
    
    @abstractmethod
    def fit(self, X, y, **kwargs) -> None:
        """Train the model on *(X, y)*."""
        ...
        
    @abstractmethod
    def predict(self, X, **kwargs) -> np.ndarray:
        """Return predictions for *X*."""
        ...

    @abstractmethod
    def save(self, path: Path) -> None:
        """Persist the model to *path*."""
        ... 

    @abstractmethod
    def load(self, path: Path):
        """Load model weights from *path* into this instance."""
        ...
        
    @abstractmethod
    def log_to_mlflow(self, name: str = "model") -> ModelInfo:
        """Log the model artifact to the active MLflow run and return its info."""
        ...
    
    @classmethod
    @abstractmethod
    def load_from_mlflow(cls, uri: str) -> ModelAdapter:
        """Instantiate an adapter by loading a model from an MLflow URI."""
        ...


class XGBoostAdapter(ModelAdapter):

    def __init__(self, params: dict | None = None, seed=1):
        from xgboost import XGBRegressor

        if params:
            self.model = XGBRegressor(**params, random_state=seed)
        else:
            self.model = XGBRegressor(random_state=seed)

    def fit(self, X: np.ndarray, y: np.ndarray, **kwargs):
        eval_set = kwargs.get("eval_set")
        
        if eval_set:
            self.model.fit(
                X,
                y,
                eval_set=eval_set,
                verbose=False
            )
        else:
            self.model.fit(X, y)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.model.predict(X)

    def save(self, path: Path) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        self.model.save_model(str(path))

    def load(self, path: Path):
        from xgboost import XGBRegressor
    
        self.model = XGBRegressor()
        return self.model.load_model(str(path))

    def log_to_mlflow(self, name: str = "model") -> ModelInfo:
        model_info = mlflow.xgboost.log_model(
            xgb_model=self.model,
            name="model",
            model_format="json",
        )
        return model_info

    @classmethod
    def load_from_mlflow(cls, uri: str) -> "XGBoostAdapter":
        adapter = cls()
        adapter.model = mlflow.xgboost.load_model(uri)
        return adapter

In [46]:
def validate_dataset_version(base_path, registry_path, version) -> Path:
    """
    Confirm that the configured dataset version exists on disk and in the registry.
    """

    base_key, abs_base_path = get_registry_key_and_paths(registry_path, base_path)

    print(base_key, abs_base_path, version)

    dataset_path = Path(f"{abs_base_path}/{version}")
    
    # dataset_path = base_key / version

    if not dataset_path.exists():

        raise RuntimeError(
            f"Dataset version {version} does not exist at {dataset_path}"
        )

    registry = read_registry(registry_path)

    dataset_registry = registry.get("datasets", {}).get(base_key, {})

    if version not in dataset_registry:

        raise RuntimeError(
            f"Dataset version {version} not registered for dataset {base_key}"
        )

    return dataset_path

def read_registry(regsitry_path):

    if regsitry_path.exists():

        data = yaml.safe_load(regsitry_path.read_text())

        if data is None:
            data = {"datasets": {}}

        return data

    else:

        return {"datasets": {}}

In [47]:
def list_fold_dirs(dataset_path: Path) -> list[Path]:
    """Return sorted fold directories inside *dataset_path*."""
    
    fold_dirs = sorted(
        [p for p in dataset_path.iterdir() if p.is_dir() and p.name.startswith("fold_")]
    )

    if not fold_dirs:
        raise RuntimeError(f"No fold directories found in {dataset_path}")

    return fold_dirs

In [48]:
def load_dataset_splits(dataset_metadata: DatasetMetadata, dataset_path: Path) -> tuple:

    schema =  dataset_metadata.dataset_schema

    trainX = pd.read_parquet(dataset_path / "train_X.parquet")
    trainY = pd.read_parquet(dataset_path / "train_y.parquet")

    valX = pd.read_parquet(dataset_path / "val_X.parquet")
    valY = pd.read_parquet(dataset_path / "val_y.parquet")

    testX = pd.read_parquet(dataset_path / "test_X.parquet")
    testY = pd.read_parquet(dataset_path / "test_y.parquet")

    sensor_bases = [
        col
        for sensor_cols in schema.sensors.values()
        for col in sensor_cols
    ]

    sensor_columns = [
        c for c in trainX.columns
        if any(c.startswith(base) for base in sensor_bases)
    ]

    import re

    def clean_feature_names(df):
        df = df.copy()
        df.columns = [
            re.sub(r"[^\w]+", "_", c).strip("_")
            for c in df.columns
        ]
        return df

    trainX = clean_feature_names(trainX)
    valX = clean_feature_names(valX)
    testX = clean_feature_names(testX)

    X_train = trainX[sensor_columns]
    X_val = valX[sensor_columns]
    X_test = testX[sensor_columns]

    # Use all columns from y parquets — works whether log-fractions is enabled or not
    y_train = trainY
    y_val = valY
    y_test = testY

    return X_train, X_val, X_test, y_train, y_val, y_test

In [49]:
def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w") as f:
        json.dump(payload, f, indent=2, default=str)

In [50]:
def log_fractions_inv_transform(pred_df: pd.DataFrame, log_fractions_meta: dict) -> pd.DataFrame:
    """
    Invert the log-fractions transform applied during preprocessing.

    Reconstructs physical concentrations from (total_ug_per_l, log_fractions_<t>...):
      predicted PFT  = total * exp(log_fractions_<t>)
      denominator    = total − sum(predicted PFTs)   (clipped to 0)

    Works on both predictions and ground-truth DataFrames.
    """
    original_targets = log_fractions_meta["original_targets"]
    denominator = log_fractions_meta["denominator"]
    numerators = [t for t in original_targets if t != denominator]

    total = pred_df["total_ug_per_l"].clip(lower=0)

    result = pd.DataFrame(index=pred_df.index)
    for num in numerators:
        result[num] = (total * np.exp(pred_df[f"log_fractions_{num}"])).clip(lower=0)

    result[denominator] = (total - result[numerators].sum(axis=1)).clip(lower=0)

    return result[original_targets]

In [93]:
class Trainer:
    """
    Orchestrates k-fold model training, optional Optuna tuning, and evaluation.

    MLflow run structure
    --------------------
    parent_run  (one per call to .train())
      ├── fold_YYYY           nested run per fold
      │     └── trial_0   nested run per Optuna trial (only when tuning enabled)
      │     └── trial_1
      │     └── ...
      └── final_test          nested run for held-out year evaluation

    Metric naming convention
    ------------------------
    All metrics follow:  {split}/{target}/{metric}
    e.g.  val/diatoms_ug_per_l/rmse

    Log-fractions mode
    ------------------
    When the dataset was built with the log-fractions target transform, y parquets contain
    (total_ug_per_l, log_fractions_<t1>, log_fractions_<t2>) instead of raw concentrations.
    The Trainer detects this via log_fractions_meta.json and back-transforms all
    predictions and ground-truth labels to physical units before computing
    metrics, so all logged values are in µg/L regardless of whether the transform is active.
    """

    def __init__(self, model_cls: type[ModelAdapter], config: AppConfig):
        self.model_cls = model_cls
        self.config = config

        storage = config.training.input_dataset

        self.dataset_path = validate_dataset_version(
            storage.base_path,
            storage.registry_path,
            storage.version,
        )

        self.dataset_meta = load_dataset_metadata(self.dataset_path)
        
        self.fold_dirs = list_fold_dirs(self.dataset_path)

        self._validate_metrics()

        self.output_root = config.training.output_dir
        self.output_root.mkdir(parents=True, exist_ok=True)

        # Load log-fractions metadata if this dataset version used the transform
        log_fractions_meta_path = self.dataset_path / "log_fractions_meta.json"
        self.log_fractions_meta = json.load(log_fractions_meta_path.open()) if log_fractions_meta_path.exists() else None
        if self.log_fractions_meta:
            print(f"log-fractions mode  |  parquet targets : {self.log_fractions_meta['log_fractions_targets']}")
            print(f"                    |  physical targets: {self.log_fractions_meta['original_targets']}")

        print(f"Training on dataset version: {self.config.training.input_dataset.version}")
        print(f"Folds found: {[d.name for d in self.fold_dirs]}")


    def _generate_run_id(self) -> str:
        """Return a timestamped, unique run ID string."""
        return "".join(["mlrun_", datetime.now().astimezone().strftime("%Y%m%d_%H%M%S_%z")])

    def _validate_metrics(self) -> None:
        for metric_name in self.config.training.metrics:
            if metric_name not in METRIC_REGISTRY:
                raise ValueError(f"Unsupported metric: {metric_name}")


    def train(self):

        mlflow.set_experiment(self.config.training.experiment_name)

        run_id = self._generate_run_id()
        run_dir = self.output_root / run_id
        run_dir.mkdir(parents=True, exist_ok=True)

        with mlflow.start_run(run_name=run_id) as parent_run:

            fold_results: dict[str, Any] = {
                "trainer_run_id": run_id,
                "dataset_version": self.config.training.input_dataset.version,
                "dataset_path": str(self.dataset_path),
            }

            # Log full config as a readable artifact
            mlflow.log_dict(self.config.model_dump(mode="json"), "config.json")

            config_path = run_dir / "config.yml"
            
            with open(config_path, "w") as f:
                yaml.safe_dump(
                    self.config.model_dump(mode="json"),
                    f,
                    sort_keys=False,
                )

            mlflow.log_artifact(config_path)
            

            # Log top-level params visible in the run comparison table
            mlflow.log_params({
                "dataset_version": self.config.training.input_dataset.version,
                "dataset_path": str(self.dataset_path),
                "model_type": self.config.training.model.type,
                "n_folds": len(self.fold_dirs),
                "tuning_enabled": bool(self.config.tuning and self.config.tuning.enabled),
                "targets": ", ".join(self.dataset_meta.dataset_schema.targets),
                "log_fractions_enabled": self.log_fractions_meta is not None,
            })

            # Per-fold training
            for fold_path in self.fold_dirs:
                fold_name = fold_path.name
                fold_results[fold_name] = self._train_fold(fold_name, fold_path, run_dir)

            # Best model selection
            best_model = self._select_best_model(fold_results)

            # Final held-out year evaluation
            final_test_results = self._final_test(fold_results, run_dir)

            summary = {
                "trainer_run_id":  run_id,
                "dataset_version": self.config.training.input_dataset.version,
                "dataset_path":    str(self.dataset_path),
                "folds":           fold_results,
                "best_model":      best_model,
                "final_test":      final_test_results,
            }

            summary_path = run_dir / "run_summary.json"
            write_json(summary_path, summary)
            mlflow.log_artifact(str(summary_path))

        return summary


    def _train_fold(self, fold_name: str, fold_path: Path, run_dir: Path) -> dict:

        with mlflow.start_run(run_name=fold_name, nested=True):

            X_train, X_val, X_test, y_train, y_val, y_test = load_dataset_splits(
                self.dataset_meta, fold_path
            )

            print(f"\nTraining {fold_name}  |  train={len(X_train)}  val={len(X_val)}  test={len(X_test)}")

            fit_kwargs = {"eval_set": [(X_val, y_val)]}

            # Hyperparameter Tuning (optional)
            if self.config.tuning and self.config.tuning.enabled:
                best_params = self._tune(X_train, X_val, y_train, y_val, fit_kwargs)
            else:
                best_params = self.config.training.model.params

            mlflow.log_params(best_params)

            model = self.model_cls(best_params, seed=self.config.project.seed)
            model.fit(X_train, y_train, **fit_kwargs)

            fold_dir = run_dir / fold_name
            fold_dir.mkdir(parents=True, exist_ok=True)
            model_path = fold_dir / self.config.training.model.save_path
            model.save(model_path)
            model_info = model.log_to_mlflow("model")
            model_uri = f"models:/{model_info.model_id}"

            year = int(fold_name.split("_")[1])

            train_metrics = self._evaluate_model(
                model_uri, X_train, y_train, run_dir=fold_dir, split="train", step=year
            )
            val_metrics = self._evaluate_model(
                model_uri, X_val, y_val, run_dir=fold_dir, split="validation", step=year
            )
            test_metrics = self._evaluate_model(
                model_uri, X_test, y_test, run_dir=fold_dir, split="test", step=year
            )

            fold_summary = {
                "fold_name":          fold_name,
                "dataset_version":    self.config.training.input_dataset.version,
                "dataset_path":       str(self.dataset_path),
                "fold_dataset_path":  str(fold_path),
                "model_uri":          model_uri,
                "model_path":         str(model_path),
                "model_info":         model_info,
                "params":             best_params,
                "train_metrics":      train_metrics,
                "val_metrics":        val_metrics,
                "test_metrics":       test_metrics,
                "fit_kwargs": {
                    k: v if k != "eval_set" else "provided"
                    for k, v in fit_kwargs.items()
                },
            }

            summary_path = fold_dir / "fold_summary.json"
            write_json(summary_path, fold_summary)
            mlflow.log_artifact(str(summary_path))

            return fold_summary


    def _tune(self, X_train, X_val, y_train, y_val, fit_kwargs) -> dict:

        metric_name    = self.config.tuning.objective_metric
        metric_fn      = METRIC_REGISTRY[metric_name]
        aggregation    = getattr(self.config.tuning, "objective_aggregation", "mean")
        target_weights = getattr(self.config.tuning, "target_weights", None)
        direction      = self.config.tuning.direction

        # Resolve physical target names for metric reporting
        physical_targets = (
            self.log_fractions_meta["original_targets"] if self.log_fractions_meta else list(y_val.columns)
        )

        def objective(trial: optuna.Trial):

            sampled_params = {}
            for name, bounds in self.config.tuning.search_space.items():
                if (
                    isinstance(bounds, list)
                    and len(bounds) == 2
                    and all(isinstance(v, int) for v in bounds)
                ):
                    sampled_params[name] = trial.suggest_int(name, bounds[0], bounds[1])
                elif (
                    isinstance(bounds, list)
                    and len(bounds) == 2
                    and all(isinstance(v, (int, float)) for v in bounds)
                    and any(isinstance(v, float) for v in bounds)
                ):
                    sampled_params[name] = trial.suggest_float(name, bounds[0], bounds[1])
                else:
                    sampled_params[name] = trial.suggest_categorical(name, bounds)

            with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):

                mlflow.log_params(sampled_params)
                mlflow.log_param("objective_aggregation", aggregation)

                model = self.model_cls(sampled_params, seed=self.config.project.seed)
                model.fit(X_train, y_train, **fit_kwargs.copy())
                preds = model.predict(X_val)
                pred_df = pd.DataFrame(preds, columns=y_val.columns, index=y_val.index)

                # Back-transform to physical space before computing objective
                if self.log_fractions_meta:
                    pred_df = log_fractions_inv_transform(pred_df, self.log_fractions_meta)
                    y_val_phys = log_fractions_inv_transform(y_val, self.log_fractions_meta)
                else:
                    y_val_phys = y_val

               # Compute per-target scores explicitly.
                # Because sklearn is silently doing multioutput averaging,
                # which cannot be overridden from outside.
                per_target_scores = {
                    target: metric_fn(y_val_phys[target], pred_df[target])
                    for target in physical_targets
                }

                for target, t_score in per_target_scores.items():
                    mlflow.log_metric(f"tuning/{target}/{metric_name}", t_score)

                # Aggregate into a single Optuna objective
                score = self._aggregate_target_scores(
                    per_target_scores, aggregation, target_weights, direction
                )
                mlflow.log_metric(f"tuning/objective/{metric_name}", score)

            mlflow.log_metric(f"tuning/objective/{metric_name}", score, step=trial.number)
            for target, t_score in per_target_scores.items():
                mlflow.log_metric(f"tuning/{target}/{metric_name}", t_score, step=trial.number)

            return score

        # TODO: If needed, change this to NSGAIISampler when there are Multi-objectives
        sampler = optuna.samplers.TPESampler(
            seed=self.config.project.seed
        )
        
        study = optuna.create_study(
            direction=self.config.tuning.direction,
            sampler=sampler
        )
        
        study.optimize(objective, n_trials=self.config.tuning.n_trials)

        print(f"  Best trial: {study.best_params}  ({metric_name}={study.best_value:.4f}, aggregation={aggregation})")

        return study.best_params


    def _aggregate_target_scores(
        self,
        per_target_scores: dict,
        aggregation: str,
        target_weights: dict | None,
        direction: str,
    ) -> float:
        """
        Collapse per-target metric scores into a single Optuna objective value.

        aggregation options
        -------------------
        mean     : simple unweighted average across targets (default).
                   Valid when all targets share the same scale and unit.

        weighted : weighted average using target_weights from config.
                   Weights do not need to sum to 1 — they are normalised here.
                   Use this when some targets matter more than others.

        worst    : returns the worst-performing target score.
                   For minimize metrics: worst = max score.
                   For maximize metrics: worst = min score.
                   Forces Optuna to keep improving the weakest target rather
                   than trading it off against stronger ones.
        """
        scores  = list(per_target_scores.values())
        targets = list(per_target_scores.keys())

        if not scores:
            raise ValueError("per_target_scores is empty — cannot aggregate.")
 
        if aggregation == "mean":
            return sum(scores) / len(scores)

        elif aggregation == "worst":
            return max(scores) if direction == "minimize" else min(scores)

        elif aggregation == "weighted":
            if not target_weights:
                raise ValueError(
                    "objective_aggregation is 'weighted' but target_weights is not "
                    "set in config. Add a target_weights dict under tuning: in config.yml."
                )
            missing = set(targets) - set(target_weights)
            if missing:
                raise ValueError(
                    f"target_weights is missing entries for: {missing}. "
                    f"All targets must have a weight: {targets}"
                )
            total_w = sum(target_weights[t] for t in targets)
            if total_w <= 0:
                raise ValueError("target_weights must sum to a positive number.")
            return sum(per_target_scores[t] * target_weights[t] for t in targets) / total_w

        else:
            raise ValueError(
                f"Unknown objective_aggregation: '{aggregation}'. "
                f"Valid options: 'mean', 'weighted', 'worst'."
            )


    def _evaluate_model(
        self,
        model_uri: str,
        X: pd.DataFrame,
        y: pd.DataFrame,
        run_dir: Path,
        split: str,
        step: int | None = None,
    ) -> dict[str, dict[str, float]]:
        """
        Evaluate one model on one data split.

        When log-fractions transform is enabled, both predictions and ground truth are
        back-transformed to physical units (µg/L) before metrics are computed and logged.

        Returns
        -------
        metrics : {target: {metric_name: value}}  — always in physical units
        """
        model = self.model_cls.load_from_mlflow(model_uri).model
        preds = model.predict(X)
        
        if preds.shape[1] != len(y.columns):
            raise ValueError("Prediction output shape does not match number of targets")
            
        pred_df = pd.DataFrame(preds, columns=y.columns, index=X.index)

        # Back-transform to physical units before metrics
        if self.log_fractions_meta:
            pred_df = log_fractions_inv_transform(pred_df, self.log_fractions_meta)
            y = log_fractions_inv_transform(y, self.log_fractions_meta)

        metrics: dict[str, dict[str, float]] = {}

        for col in y.columns:
            metrics[col] = {}
            for metric_name in self.config.training.metrics:
                fn = METRIC_REGISTRY[metric_name]
                value = fn(y[col], pred_df[col])
                metrics[col][metric_name] = value
                mlflow.log_metric(f"{split}/{col}/{metric_name}", value, step=step)

        # mlflow.log_dict(metrics, f"{run_dir}/{split}_metrics.json")

        metrics_file = run_dir / f"{split}_metrics.json"

        metrics_file.parent.mkdir(parents=True, exist_ok=True)

        with open(metrics_file, "w") as f:
            json.dump(metrics, f, indent=2)

        artifact_dir = split if step is None else f"{split}/{step}"

        mlflow.log_artifact(
            str(metrics_file),
            artifact_path=artifact_dir,
        )
                
        # mlflow.log_artifact(str(metrics_file))

        split_tag = split if step is None else f"{split}/{step}"
        self._log_prediction_csv(run_dir, split_tag, X, y, pred_df)
        self._log_prediction_plot(run_dir, split_tag, y, pred_df)
        self._log_shap(run_dir, model, X, y.columns, split_tag)

        return metrics

    def _evaluate_ensemble(
        self,
        name: str,
        y_true: pd.DataFrame,
        y_pred: pd.DataFrame,
        X: pd.DataFrame,
        run_dir: Path,
    ) -> dict[str, dict[str, float]]:
        """Evaluate and log an ensemble prediction (expects physical-unit inputs)."""
        metrics: dict[str, dict[str, float]] = {}

        for col in y_true.columns:
            metrics[col] = {}
            for metric_name in self.config.training.metrics:
                fn    = METRIC_REGISTRY[metric_name]
                value = fn(y_true[col], y_pred[col])
                metrics[col][metric_name] = value
                mlflow.log_metric(f"{name}/{col}/{metric_name}", value)

        # mlflow.log_dict(metrics, f"{name}_metrics.json")

        metrics_file = run_dir / f"{name}_metrics.json"

        metrics_file.parent.mkdir(parents=True, exist_ok=True)

        with open(metrics_file, "w") as f:
            json.dump(metrics, f, indent=2)
        
        mlflow.log_artifact(str(metrics_file), artifact_path=name)

        split_tag = f"final_test/{name}"

        self._log_prediction_csv(run_dir, split_tag, X, y_true, y_pred)
        self._log_prediction_plot(run_dir, split_tag, y_true, y_pred)

        return metrics


    def _log_prediction_csv(
        self,
        run_dir: Path,
        split: str,
        X: pd.DataFrame,
        y_true: pd.DataFrame,
        y_pred: pd.DataFrame,
    ):
        df = X.copy()
        for col in y_true.columns:
            df[f"{col}_true"] = y_true[col]
            df[f"{col}_pred"] = y_pred[col]

        pred_dir = run_dir / "predictions"
        path = pred_dir / f"{split}_full_predictions.csv"
        path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(path, index=False)
        mlflow.log_artifact(str(path), artifact_path=f"{split}/predictions")


    def _log_prediction_plot(
        self,
        run_dir: Path,
        split: str,
        y_true: pd.DataFrame,
        y_pred: pd.DataFrame,
    ):
        plot_dir = run_dir / "plots"

        for col in y_true.columns:
            n = min(200, len(y_true))
            yt = y_true[col].iloc[:n]
            yp = y_pred[col].iloc[:n]

            fig, ax = plt.subplots()
            ax.scatter(yt, yp, alpha=0.6)

            # Ideal prediction line
            lim = [min(yt.min(), yp.min()), max(yt.max(), yp.max())]
            ax.plot(lim, lim, "r--", linewidth=1, label="perfect")

            ax.set_title(f"{split}-{col} prediction vs actual")
            ax.set_xlabel("Actual")
            ax.set_ylabel("Prediction")

            path = plot_dir / f"{split}_{col}_pred_vs_true.png"
            path.parent.mkdir(parents=True, exist_ok=True)
            plt.savefig(path)
            plt.close(fig)
            mlflow.log_artifact(str(path), artifact_path=f"{split}/plots")

            if split.startswith(("validation", "test", "final_test")):
                mlflow.log_artifact(str(path))

    def _log_shap(self, run_dir: Path, model, X: pd.DataFrame, targets, split: str):

        shap_dir = run_dir / "shap"
        X_sample = X.sample(min(200, len(X)), random_state=self.config.project.seed)

        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_sample)

        if isinstance(shap_values, list):
            shap_per_target = shap_values
        else:
            shap_per_target = [
                shap_values[:, :, i]
                for i in range(shap_values.shape[2])
            ]

        for i, target in enumerate(targets):
            vals = shap_per_target[i]

            # Summary plot
            shap.summary_plot(vals, X_sample, show=False)
            path = shap_dir / f"{target}_summary.png"
            path.parent.mkdir(parents=True, exist_ok=True)
            plt.savefig(path, bbox_inches="tight")
            plt.close()
            mlflow.log_artifact(str(path), artifact_path=f"{split}/shap")
    
            # Bar plot
            shap.plots.bar(
                shap.Explanation(
                    values=vals,
                    data=X_sample.values,
                    feature_names=X_sample.columns,
                ),
                show=False,
            )
            path = shap_dir / f"{target}_bar.png"
            path.parent.mkdir(parents=True, exist_ok=True)
            plt.savefig(path, bbox_inches="tight")
            plt.close()
            mlflow.log_artifact(str(path), artifact_path=f"{split}/shap")

            # Waterfall plot (first sample only).
            explanation = shap.Explanation(
                values=vals[0],
                base_values=explainer.expected_value[i],
                data=X_sample.iloc[0],
                feature_names=X_sample.columns,
            )
            shap.plots.waterfall(explanation, show=False)
            path = shap_dir / f"{target}_waterfall_0.png"
            path.parent.mkdir(parents=True, exist_ok=True)
            plt.savefig(path, bbox_inches="tight")
            plt.close()
            mlflow.log_artifact(str(path), artifact_path=f"{split}/shap")


    def _select_best_model(self, fold_results: dict) -> dict:
        """
        Select the best fold model overall AND per target based on val metrics.

        Val metrics are always in physical units (back-transformed when log-fractions
        transform is active), so selection logic is the same regardless of that setting.

        Overall selection modes (controlled by config):
          - target=None, mode='mean'  -> average metric across all targets (default)
          - target=None, mode='worst' -> worst target drives selection (conservative)
          - target='col_name'         -> single specific target

        Per-target selection always runs regardless of mode, giving you a
        separate best-fold answer for each target independently.

        Returns
        -------
        {
            "overall": { model_uri, fold, metric, score, target, mode },
            "per_target": {
                "<target_col>": { model_uri, fold, score },
                ...
            }
        }
        """
        sel = self.config.training.best_model_selection
        metric    = sel.metric
        direction = sel.direction
        target    = getattr(sel, "target", None)
        mode      = getattr(sel, "mode", "mean")

        targets = self.dataset_meta.dataset_schema.targets

        # Overall best (single winner across all targets)
        best_uri = None
        best_score = None
        best_fold = None

        for fold_name, result in fold_results.items():
            if not fold_name.startswith("fold_"):
                continue
            val_metrics = result["val_metrics"]
            if not val_metrics:
                continue

            if target is not None:
                if target not in val_metrics:
                    raise ValueError(
                        f"best_model_selection.target '{target}' not found. "
                        f"Available: {list(val_metrics)}"
                    )
                score = val_metrics[target][metric]
            elif mode == "worst":
                scores = [val_metrics[t][metric] for t in val_metrics]
                score  = max(scores) if direction == "minimize" else min(scores)
            else:
                # mean — only valid when targets share the same scale/unit
                scores = [val_metrics[t][metric] for t in val_metrics]
                score  = sum(scores) / len(scores)

            is_better = (
                best_score is None
                or (direction == "minimize" and score < best_score)
                or (direction == "maximize" and score > best_score)
            )
            if is_better:
                best_score = score
                best_uri   = result["model_uri"]
                best_fold  = fold_name

        if best_uri is None:
            raise ValueError("No valid folds found for model selection.")

        # Per-target best (independent winner for each target)
        # Answers: "which fold model was best specifically for for e.g. diatoms?"
        per_target_best: dict[str, dict] = {}

        for t in targets:
            t_best_uri   = None
            t_best_score = None
            t_best_fold  = None

            for fold_name, result in fold_results.items():
                if not fold_name.startswith("fold_"):
                    continue
                val_metrics = result.get("val_metrics", {})
                if t not in val_metrics:
                    continue

                score = val_metrics[t][metric]
                is_better = (
                    t_best_score is None
                    or (direction == "minimize" and score < t_best_score)
                    or (direction == "maximize" and score > t_best_score)
                )
                if is_better:
                    t_best_score = score
                    t_best_uri   = result["model_uri"]
                    t_best_fold  = fold_name

            per_target_best[t] = {
                "model_uri": t_best_uri,
                "fold":      t_best_fold,
                "score":     t_best_score,
            }

        print(f"\nBest model (overall, mode='{mode}'): {best_fold}  ({metric}={best_score:.4f})")
        print("Best model per target:")
        for t, info in per_target_best.items():
            print(f"  {t:40s}  {info['fold']}  ({metric}={info['score']:.4f})")

        return {
            "overall": {
                "model_uri": best_uri,
                "fold":      best_fold,
                "metric":    metric,
                "score":     best_score,
                "target":    target,
                "mode":      mode,
            },
            "per_target": per_target_best,
        }


    def _final_test(self, fold_results, run_dir):
        """
        Evaluate all fold models on the held-out final test year.

        When log-fractions transform is enabled:
          - Each fold's predictions are back-transformed to physical units before
            being combined into ensembles (averaging in physical space).
          - y_final is also back-transformed before ensemble evaluation.
          - val_metrics (used for weighting) are already in physical units.

        Three ensembles are produced:
          simple               — unweighted mean
          weighted             — per-fold inverse-val-metric weight (same across targets)
          weighted_per_target  — per-fold AND per-target inverse-val-metric weight -> each target column is weighted independently based on how well each fold model predicted *that specific target* on val
        """
        final_path = self.dataset_path / "final_test"
        X_final = pd.read_parquet(final_path / "test_X.parquet")
        y_final = pd.read_parquet(final_path / "test_y.parquet")

        sensor_bases   = [c for cols in self.dataset_meta.dataset_schema.sensors.values() for c in cols]
        sensor_columns = [c for c in X_final.columns if any(c.startswith(b) for b in sensor_bases)]
        X_final        = X_final[sensor_columns].copy()

        weight_metric = self.config.training.best_model_selection.metric
        if weight_metric not in self.config.training.metrics:
            weight_metric = self.config.training.metrics[0]
        direction = self.config.training.best_model_selection.direction

        # Physical-space targets and ground truth
        targets_physical = (
            self.log_fractions_meta["original_targets"] if self.log_fractions_meta else list(y_final.columns)
        )
        y_final_phys = (
            log_fractions_inv_transform(y_final, self.log_fractions_meta) if self.log_fractions_meta else y_final
        )

        results               = {}
        all_preds             = []
        global_weights        = []
        global_weighted_preds = []

        per_target_weights        = {t: [] for t in targets_physical}
        per_target_weighted_preds = {t: [] for t in targets_physical}

        with mlflow.start_run(run_name="final_test", nested=True):

            fold_keys = [k for k in fold_results if k.startswith("fold_")]
            mlflow.log_param("n_fold_models", len(fold_keys))
            mlflow.log_param("weight_metric", weight_metric)

            for fold_name, fold_data in fold_results.items():

                if not fold_name.startswith("fold_"):
                    continue

                model_uri = fold_data["model_uri"]
                model = self.model_cls.load_from_mlflow(model_uri).model
                preds = model.predict(X_final)
                pred_df = pd.DataFrame(preds, columns=y_final.columns, index=X_final.index)

                # Back-transform to physical space for ensemble building
                pred_df_phys = (
                    log_fractions_inv_transform(pred_df, self.log_fractions_meta) if self.log_fractions_meta else pred_df
                )
                all_preds.append(pred_df_phys)

                # Per-fold final test metrics (_evaluate_model handles back-transform internally)
                year = int(fold_name.split("_")[1])
                results[fold_name] = self._evaluate_model(
                    model_uri,
                    X_final.copy(),
                    y_final.copy(),
                    run_dir=run_dir / "final_test" / fold_name,
                    split="final_test",
                    step=year,
                )

                # Global weight: mean val score across physical targets
                val_scores     = [fold_data["val_metrics"][t][weight_metric] for t in targets_physical]
                mean_val_score = sum(val_scores) / len(val_scores)
                global_w = (
                    1.0 / (mean_val_score + 1e-8)
                    if direction == "minimize"
                    else max(mean_val_score, 1e-8)
                )
                global_weights.append(global_w)
                global_weighted_preds.append(pred_df_phys * global_w)

                # Per-target weight: val score for that specific target
                for t in targets_physical:
                    t_score = fold_data["val_metrics"][t][weight_metric]
                    t_w = (
                        1.0 / (t_score + 1e-8)
                        if direction == "minimize"
                        else max(t_score, 1e-8)
                    )
                    per_target_weights[t].append(t_w)
                    per_target_weighted_preds[t].append(pred_df_phys[t] * t_w)

            # Build ensembles (all in physical space)
            ensemble_simple   = sum(all_preds) / len(all_preds)
            ensemble_weighted = sum(global_weighted_preds) / sum(global_weights)

            # Per-target weighted: each column assembled from its own weights
            ensemble_per_target = pd.DataFrame(index=X_final.index)
            for t in targets_physical:
                total_t_w = sum(per_target_weights[t])
                ensemble_per_target[t] = sum(per_target_weighted_preds[t]) / total_t_w
 
            # Evaluate and log all three ensembles
            final_test_dir = run_dir / "final_test"
            final_test_dir.mkdir(parents=True, exist_ok=True)

            simple_metrics     = self._evaluate_ensemble("ensemble_simple", y_final_phys, ensemble_simple, X_final, final_test_dir)
            weighted_metrics   = self._evaluate_ensemble("ensemble_weighted", y_final_phys, ensemble_weighted, X_final, final_test_dir)
            per_target_metrics = self._evaluate_ensemble("ensemble_weighted_per_target", y_final_phys, ensemble_per_target, X_final, final_test_dir)

            print("\n--- Final test results ---")
            print(f"  {'target':40s}  {'metric':6s}  {'simple':>10}  {'weighted':>10}  {'per_target_w':>14}")
            for col in y_final_phys.columns:
                for m in self.config.training.metrics:
                    sv = simple_metrics[col][m]
                    wv = weighted_metrics[col][m]
                    pv = per_target_metrics[col][m]
                    print(f"  {col:40s}  {m:6s}  {sv:10.4f}  {wv:10.4f}  {pv:14.4f}")

            results["ensemble_simple"]               = simple_metrics
            results["ensemble_weighted"]             = weighted_metrics
            results["ensemble_weighted_per_target"]  = per_target_metrics

        per_fold_rows = {k: v for k, v in results.items() if k.startswith("fold_")}
        if per_fold_rows:
            df   = pd.DataFrame(per_fold_rows).T
            path = run_dir / "final_test_per_fold.csv"
            df.to_csv(path)
            mlflow.log_artifact(str(path))

        return results

In [94]:
def load_dataset_metadata(dataset_path: Path) -> DatasetMetadata:

    metadata_path = dataset_path / "metadata.yml"

    if not metadata_path.exists():
        raise FileNotFoundError(
            f"Dataset metadata not found:\n{metadata_path}"
        )

    with open(metadata_path) as f:
        raw = yaml.safe_load(f)

    return DatasetMetadata(**raw)

In [95]:
# TODO:
# outlier detection if a model's std deviation is more than threshold, ignore it from the calculation.
# maybe we try quantile regression instead of MSE for next tests of model training?

In [100]:
with open("../../pftnc-experiments/config/config.yml") as f:
    config_dict = yaml.safe_load(f)

config = AppConfig(**config_dict)

In [101]:
config

AppConfig(project=ProjectConfig(seed=42), feature_engineering=FeatureEngineeringConfig(input_dataset=DatasetInputConfig(path=None, base_path=PosixPath('/home/yogesh/Projects/BC/pftnc-data/data/simulated/elbe_bunthaus'), registry_path=PosixPath('/home/yogesh/Projects/BC/pftnc-data/data/simulated/elbe_bunthaus/elbe_bunthaus.yaml'), version='elbe_bunthaus_20260528T152409Z_ad40893c_3a8b37'), output_dataset=DatasetSpec(storage=DatasetOutputConfig(base_path=PosixPath('/home/yogesh/Projects/BC/pftnc-data/data/feature_engineered/elbe_bunthaus'), registry_path=PosixPath('/home/yogesh/Projects/BC/pftnc-data/data/feature_engineered/elbe_bunthaus/elbe_bunthaus.yaml'), description='First version of feature engineered dataset with default settings below.'), metadata=DatasetMetadata(sites=['elbe_bunthaus'], dataset_schema=DatasetSchema(date_column='time', group_column='site', sensors={'chime': ['chime_diatoms_ug_per_l', 'chime_cyanobacteria_ug_per_l', 'chime_others_ug_per_l'], 'lstm': ['lstm_lswt_c']

In [102]:
def set_global_seed(seed: int):
    import random
    random.seed(seed)
    np.random.seed(seed)


seed = config.project.seed
set_global_seed(seed)

print(f"Global seed set to {seed}")

Global seed set to 42


### Joining Steps FE and training below

In [24]:
# outout from FE step

fe_result = DatasetArtifact(base_path=Path('data/feature_engineered'), registry_path=Path('data/feature_engineered.yaml'), version='elbe_bunthaus_20260513T111150Z_b694b321_1c3e69')
fe_result

DatasetArtifact(base_path=PosixPath('data/feature_engineered'), registry_path=PosixPath('data/feature_engineered.yaml'), version='elbe_bunthaus_20260513T111150Z_b694b321_1c3e69')

In [25]:
input_training = fe_result.to_input_config()
input_training

DatasetInputConfig(path=None, base_path=PosixPath('data/feature_engineered'), registry_path=PosixPath('data/feature_engineered.yaml'), version='elbe_bunthaus_20260513T111150Z_b694b321_1c3e69')

In [26]:
## Set the output of Feature engineering to input of training dynamically here. If training is run standalone, without FE, then it should take what is provided in the config, if both FE and training are run together, update the input_dataset object dynamically as shown below...

config.training.input_dataset = input_training

In [103]:
%%time
trainer = Trainer(
    model_cls=XGBoostAdapter,
    config=config
)

results = trainer.train()

print("Final Results:")
print(results)

[I 2026-06-05 16:36:50,768] A new study created in memory with name: no-name-4e1fbb00-2a0a-4329-8aa0-aa9314a610e7


data/feature_engineered/elbe_bunthaus /home/yogesh/Projects/BC/pftnc-data/data/feature_engineered/elbe_bunthaus elbe_bunthaus_20260528T162627Z_ac8dd97e_979fbd
log-fractions mode  |  parquet targets : ['total_ug_per_l', 'log_fractions_diatoms_ug_per_l', 'log_fractions_cyanobacterial_ug_per_l']
                    |  physical targets: ['diatoms_ug_per_l', 'cyanobacterial_ug_per_l', 'others_ug_per_l']
Training on dataset version: elbe_bunthaus_20260528T162627Z_ac8dd97e_979fbd
Folds found: ['fold_2015', 'fold_2016', 'fold_2017', 'fold_2018', 'fold_2019', 'fold_2021', 'fold_2022', 'fold_2024']

Training fold_2015  |  train=1287  val=267  test=271


[I 2026-06-05 16:36:51,136] Trial 0 finished with value: 24.948995311061424 and parameters: {'n_estimators': 5, 'learning_rate': 0.01, 'max_depth': 4, 'subsample': 0.8, 'colsample_bytree': 1.0, 'reg_lambda': 5, 'reg_alpha': 10}. Best is trial 0 with value: 24.948995311061424.


🏃 View run trial_0 at: http://127.0.0.1:8081/#/experiments/1/runs/168c2db3213f40a0a0498134d7f2a99d
🧪 View experiment at: http://127.0.0.1:8081/#/experiments/1


[I 2026-06-05 16:36:51,547] Trial 1 finished with value: 23.774079415290554 and parameters: {'n_estimators': 2, 'learning_rate': 0.1, 'max_depth': 10, 'subsample': 1.0, 'colsample_bytree': 1.0, 'reg_lambda': 5, 'reg_alpha': 10}. Best is trial 1 with value: 23.774079415290554.


...

/tmp/ipykernel_336143/690697286.py:536: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(vals, X_sample, show=False)
/tmp/ipykernel_336143/690697286.py:536: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(vals, X_sample, show=False)
/tmp/ipykernel_336143/690697286.py:536: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(vals, X_sample, show=False)


/tmp/ipykernel_336143/690697286.py:536: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(vals, X_sample, show=False)
/tmp/ipykernel_336143/690697286.py:536: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(vals, X_sample, show=False)
/tmp/ipykernel_336143/690697286.py:536: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(vals, X_sample, show=False)


/tmp/ipykernel_336143/690697286.py:536: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(vals, X_sample, show=False)
/tmp/ipykernel_336143/690697286.py:536: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(vals, X_sample, show=False)
/tmp/ipykernel_336143/690697286.py:536: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(vals, X_sample, show=False)


/tmp/ipykernel_336143/690697286.py:536: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(vals, X_sample, show=False)
/tmp/ipykernel_336143/690697286.py:536: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(vals, X_sample, show=False)
/tmp/ipykernel_336143/690697286.py:536: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(vals, X_sample, show=False)



--- Final test results ---
  target                                    metric      simple    weighted    per_target_w
  diatoms_ug_per_l                          mse       172.0378    164.3330        164.2081
  diatoms_ug_per_l                          rmse       13.1163     12.8192         12.8144
  diatoms_ug_per_l                          r2          0.8297      0.8373          0.8374
  cyanobacterial_ug_per_l                   mse         7.5279      5.6864          5.8756
  cyanobacterial_ug_per_l                   rmse        2.7437      2.3846          2.4240
  cyanobacterial_ug_per_l                   r2          0.7744      0.8296          0.8239
  others_ug_per_l                           mse        87.2541     79.6176         79.6062
  others_ug_per_l                           rmse        9.3410      8.9229          8.9222
  others_ug_per_l                           r2          0.7280      0.7518          0.7518
🏃 View run final_test at: http://127.0.0.1:8081/#/experiments/